In [1]:
# Install the exact versions requested
!pip install ultralytics==8.4.14
!pip install opencv-contrib-python==4.11.0.86

import os
import zipfile
import yaml
import torch
import pandas as pd
from ultralytics import YOLO

# 1. Unzip the dataset
zip_path = '/content/drive/MyDrive/Colab Notebooks/Weeds.v3-augmented_nottrained.yolov8.zip'
extract_path = '/content/weed_data'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(" Dataset extracted.")
else:
    print(" ZIP file not found! Please upload it to the Files tab.")

# 2. Fix the data.yaml paths for Colab
yaml_path = os.path.join(extract_path, 'data.yaml')
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    data_config['train'] = os.path.join(extract_path, 'train/images')
    data_config['val'] = os.path.join(extract_path, 'valid/images')
    data_config['test'] = os.path.join(extract_path, 'test/images')

    with open(yaml_path, 'w') as f:
        yaml.dump(data_config, f)
    print(" data.yaml paths updated.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 15.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-contrib-python
    Found existing installation: opencv-contrib-python 4.13.0.92
    Uninstalling opencv-contrib-python-4.13.0.92:
      Successfully uninstalled opencv-contrib-python-4.13.0.92
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset extracted.
✅ data.yaml paths updated.


In [2]:
# Block 1: Training
model_variants = ['yolov8n.pt']
device_to_use = 0 if torch.cuda.is_available() else 'cpu'

for model_file in model_variants:
    model_name = model_file.replace('.pt', '')
    print(f" Training {model_name}...")

    model = YOLO(model_file)
    model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        name=f"weed_{model_name}",
        device=device_to_use
    )

🚀 Training yolov8n...
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weed_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

In [3]:
# Block 2: Evaluation & Data Collection
results_list = []

for model_file in model_variants:
    model_name = model_file.replace('.pt', '')
    # Point to where YOLO saved the best weights
    best_weight_path = f"runs/detect/weed_{model_name}/weights/best.pt"

    if os.path.exists(best_weight_path):
        # Load the trained model
        model = YOLO(best_weight_path)

        # Run validation
        metrics = model.val()

        # Calculate file size
        size_mb = os.path.getsize(best_weight_path) / (1024 * 1024)

        results_list.append({
            "Model": f"YOLOv8{model_name[-1]}",
            "mAP50": metrics.results_dict['metrics/mAP50(B)'],
            "mAP50-95": metrics.results_dict['metrics/mAP50-95(B)'],
            "Precision": metrics.results_dict['metrics/precision(B)'],
            "Recall": metrics.results_dict['metrics/recall(B)'],
            "Speed (ms)": metrics.speed['inference'],
            "Size (MB)": round(size_mb, 2)
        })
    else:
        print(f" Warning: Weights not found for {model_name}")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1883.8±770.9 MB/s, size: 62.8 KB)
val: Scanning /content/weed_data/valid/labels.cache... 359 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 359/359 136.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 3.8it/s 6.1s
                   all        359        920      0.936       0.96      0.976      0.762
Speed: 2.8ms preprocess, 4.9ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/runs/detect/val


In [4]:
# Block 3: Display Results
if results_list:
    comparison_df = pd.DataFrame(results_list)
    print("\n--- FINAL COMPARISON TABLE ---")
    print(comparison_df.to_string(index=False))

    # Optional: Save to CSV so you don't lose the data
    comparison_df.to_csv("model_comparison_results.csv", index=False)


--- FINAL COMPARISON TABLE ---
  Model    mAP50  mAP50-95  Precision   Recall  Speed (ms)  Size (MB)
YOLOv8n 0.975736  0.761755   0.936206 0.959783    4.867258       5.96
